# Test of forager index conversation

When forager indices are non-consecutive (i.e., [0, 2]), the dataObject internally converts these indices to consecutive (i.e., [0,1]) and stores the mapping within two variables:
- local_to_global_map: mapping back to original, non-consecutive indices
- global_to_local_map: mapping to consecutive indices (applied internally at ```__init__``` when creating dataObject)  

In [23]:
import os

import pandas as pd

import collab.foraging.toolkit as ftk

smoke_test = "CI" in os.environ

num_svi_iters = 400 if not smoke_test else 4
num_samples = 1000 if not smoke_test else 10

### Load data

Setup for dataObject with ground-truth indices

In [24]:
# load data
fish_data = pd.read_csv("4wpf_test.csv")
gridMin = 0
gridMax = 300
grid_size = 50

# scaling and subsampling
fishDF_scaled = ftk.rescale_to_grid(
    fish_data, size=grid_size, gridMin=gridMin, gridMax=gridMax
)
# create a test foragers object with 20 frames
num_frames = 10
foragers_object = ftk.dataObject(
    fishDF_scaled,
    grid_size=grid_size,
)

/Users/tommybotch/Documents/collab-creatures/collab/foraging/toolkit/utils.py:42: UserWarning: 
                NaN values in data. The default behavior of predictor/score generating functions is
                to ignore foragers with missing positional data. To modify, see documentation of
                `derive_predictors_and_scores` and `generate_local_windows`
                
  warnings.warn(


### Modify the original dataframe

We subsample forager indices to test the global (non-consecutive) to local (consecutive) mapping

In [26]:
fishDF_nonconsecutive = fishDF_scaled[fishDF_scaled["forager"].isin([0, 2])]

# Given non-consecutive indices, forager indices are converted to consecutive integers and a warning is raised
foragers_nonconsecutive_obj = ftk.dataObject(fishDF_nonconsecutive, grid_size=grid_size)

/Users/tommybotch/Documents/collab-creatures/collab/foraging/toolkit/utils.py:42: UserWarning: 
                NaN values in data. The default behavior of predictor/score generating functions is
                to ignore foragers with missing positional data. To modify, see documentation of
                `derive_predictors_and_scores` and `generate_local_windows`
                
  warnings.warn(
/Users/tommybotch/Documents/collab-creatures/collab/foraging/toolkit/utils.py:59: UserWarning: 
                Original forager indices were converted to consecutive integers starting from 0.
                To access the original forager IDs, use the apply_forager_id_mapping() method.
                Original IDs were: [0 2]
                
  warnings.warn(


### Demonstrate application of local-global maps

Whether mapping is needed is stored in a read-only property ```needs_forager_id_mapping```

In [27]:
# Original object
print(f"Original DF needed mapping: {foragers_object.needs_forager_id_mapping}")

# Non-consecutive object
print(
    f"Non-consecutive DF needed mapping: {foragers_nonconsecutive_obj.needs_forager_id_mapping}"
)

# Maps are accessible as properties
# Local to global = maps back to original indices
# Global to local = maps back to consecutive indices
print(f"Local to global map: {foragers_nonconsecutive_obj.local_to_global_map}")
print(f"Global to local map: {foragers_nonconsecutive_obj.global_to_local_map}")

Original DF needed mapping: False
Non-consecutive DF needed mapping: True
Local to global map: {0: 0, 1: 2}
Global to local map: {0: 0, 2: 1}
